In [1]:
import pandas as pd
import numpy as np
import os
import joblib
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor

In [4]:
MODEL_FILE = "model.pkl"
PIPELINE_FILE = "pipe.pkl"

def build_pipeline(num_attribs,cat_attribs):
    num_pipe = Pipeline([
        ("impute",SimpleImputer(strategy="median")),
        ("scaler",StandardScaler())
    ])
    
    cat_pipe = Pipeline([
        ("ohe",OneHotEncoder(handle_unknown="ignore"))
    ])

    full_pipe = ColumnTransformer([
        ("num",num_pipe,num_attribs),
        ("cat",cat_pipe,cat_attribs)
    ])

    return full_pipe

if not os.path.exists(MODEL_FILE):
    #Lets train the model
    data = pd.read_csv('housing.csv')
    
    x = data.drop('median_house_value',axis= True)
    y = data['median_house_value'].copy()

    num_attribs = x.drop("ocean_proximity",axis=1).columns.to_list()
    cat_attribs = ['ocean_proximity']

    #pipeline 
    pipeline = build_pipeline(num_attribs,cat_attribs)
    x_prepared = pipeline.fit_transform(x)

    all_column = ['longitude','latitude','housing_median_age','total_rooms',
              'total_bedrooms','population','households','median_income',
              "<1H OCEAN","INLAND","ISLAND","NEAR BAY","NEAR OCEAN"]

    x_prepared = pd.DataFrame(x_prepared,columns=all_column,index=data.index)

    # Building a Random Forest Regression Model on data
    x_train,x_test,y_train,y_test = train_test_split(x_prepared,y,test_size=0.2,random_state=42)

    model = RandomForestRegressor(random_state=42)
    model.fit(x_train,y_train)

    joblib.dump(model,MODEL_FILE)
    joblib.dump(pipeline,PIPELINE_FILE)
    print("Model is trained, Congrats!!")

else:
    #Let's do inference
    model = joblib.load(MODEL_FILE)
    pipeline = joblib.load(PIPELINE_FILE)

    # input_data = pd.read_csv('input.csv') 
    # transformed_input = pipeline.transform(input_data)
    # predictions = model.predict(transformed_input)
    # input_data['median_house_value'] = predictions

    # input_data.to_csv("output.csv",index=False)
    # print("Result is saved to output.csv !!!")
    

Model is trained, Congrats!!


In [65]:
x_train1,x_test1,y_train1,y_test1 = train_test_split(x,y,test_size=0.2,random_state=42).copy()

original_test_index = x_test1.index

x_test1 = x_test1.drop('ocean_proximity',axis=1)
imput = SimpleImputer(strategy="median")
x_test1 = imput.fit_transform(x_test1)

x_test1 = pd.DataFrame(x_test1,columns=['longitude','latitude','housing_median_age','total_rooms',
                                        'total_bedrooms','population','households','median_income'],index=original_test_index)
x_test1['ocean_proximity'] = data['ocean_proximity']



20046     47700.0
3024      45800.0
15663    500001.0
20484    218600.0
9814     278000.0
           ...   
15362    263300.0
16623    266800.0
18086    500001.0
2144      72300.0
3665     151500.0
Name: median_house_value, Length: 4128, dtype: float64

In [66]:
test_data = pd.concat([x_test1,y_test1],axis=1)
test_data.to_csv('input.csv',index=False)
print("Csv Created successfuly!")

Csv Created successfuly!
